# Fine-tuning do Swin Transformer para classificação de engajamento

Este notebook faz o fine-tuning do `microsoft/swin-base-patch4-window7-224-in22k` usando as imagens de rosto extraídas pelo `extract_faces.py`, organizadas em:

```
data/
  engajado/
  nao_engajado/
```

Antes de rodar este notebook, baixe o modelo base e salve localmente com:

```
python download_base_model.py
```

Isso cria a pasta `swin-base-local/` (processor + pesos), usada abaixo em `MODEL_NAME` em vez de baixar direto do Hugging Face Hub a cada execução.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, Features, ClassLabel, Image as HFImage
from sklearn.model_selection import train_test_split
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## Configuração

In [ ]:
MODEL_NAME = "swin-base-local"  # pasta local gerada por download_base_model.py
DATA_DIR = "data"  # pasta com subpastas engajado/ e nao_engajado/
OUTPUT_DIR = "swin-engajamento-finetuned"
TEST_SIZE = 0.2
SEED = 42

BATCH_SIZE = 8
NUM_EPOCHS = 5
LEARNING_RATE = 5e-5

## Carregar o dataset

Em vez de usar `load_dataset("imagefolder", ...)`, montamos o dataset manualmente percorrendo as pastas com `pathlib`. Isso evita um bug de incompatibilidade entre `datasets`/`fsspec` (`ValueError: Invalid pattern: '**' can only be an entire path component`) que ocorre em alguns ambientes (ex: SageMaker) quando essas duas ficam em versões desencontradas.

O split treino/validação estratificado também é feito manualmente com `sklearn.model_selection.train_test_split`, já que o `Dataset.train_test_split(..., stratify_by_column=...)` só existe em versões mais novas do `datasets` (o SageMaker aqui está com a versão 2.2.1).

Cada subpasta de `data/` vira uma classe (label).

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}


def build_image_dataset(data_dir):
    data_dir = Path(data_dir)
    class_names = sorted(d.name for d in data_dir.iterdir() if d.is_dir())

    image_paths, label_names = [], []
    for class_name in class_names:
        for file in sorted((data_dir / class_name).iterdir()):
            if file.suffix.lower() in IMAGE_EXTENSIONS:
                image_paths.append(str(file))
                label_names.append(class_name)

    features = Features({"image": HFImage(), "label": ClassLabel(names=class_names)})
    return Dataset.from_dict({"image": image_paths, "label": label_names}, features=features)


full_dataset = build_image_dataset(DATA_DIR)

indices = list(range(len(full_dataset)))
train_idx, val_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=SEED, stratify=full_dataset["label"]
)
train_ds = full_dataset.select(train_idx)
val_ds = full_dataset.select(val_idx)

labels = full_dataset.features["label"].names
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}

print(f"Classes: {labels}")
print(f"Treino: {len(train_ds)} imagens | Validação: {len(val_ds)} imagens")

## Pré-processamento

Usamos o mesmo `AutoImageProcessor` do modelo base para redimensionar/normalizar as imagens no formato esperado pelo Swin. A transformação é aplicada "on the fly" (`set_transform`), ou seja, só roda quando o exemplo é lido durante o treino.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)


def transform(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    batch["pixel_values"] = inputs["pixel_values"]
    return batch


train_ds.set_transform(transform)
val_ds.set_transform(transform)

## Carregar o modelo

`ignore_mismatched_sizes=True` é necessário porque a cabeça de classificação original do checkpoint (treinada para 21k+ classes) é trocada por uma nova cabeça com `len(labels)` saídas (2, no nosso caso).

In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

## Métricas e collate function

In [ ]:
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels_batch = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels_batch}


def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(label_ids, preds),
        "f1": f1_score(label_ids, preds, average="weighted"),
        "precision": precision_score(label_ids, preds, average="weighted", zero_division=0),
        "recall": recall_score(label_ids, preds, average="weighted"),
    }

## Treinamento

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    remove_unused_columns=False,
    dataloader_num_workers=0,  # evita problemas de multiprocessing no Windows/Jupyter
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [ ]:
train_result = trainer.train()
trainer.save_metrics("train", train_result.metrics)

## Avaliação final

In [ ]:
eval_metrics = trainer.evaluate()
trainer.save_metrics("eval", eval_metrics)
eval_metrics

## Salvar o modelo treinado

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Modelo salvo em: {OUTPUT_DIR}")

## Teste rápido de inferência

Pega uma imagem do conjunto de validação e mostra a predição do modelo treinado.

In [ ]:
sample = val_ds[0]
image = sample["image"].convert("RGB")
pixel_values = sample["pixel_values"].unsqueeze(0)

model.eval()
with torch.no_grad():
    logits = model(pixel_values=pixel_values).logits

pred_id = logits.argmax(-1).item()
print(f"Classe real: {id2label[sample['label']]}")
print(f"Classe predita: {id2label[pred_id]}")
image